To install OR-Tools, run the following cell:

In [1]:
!pip install ortools


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


You are given several classes for reading the instance data from a file, storing the instance data and returning a solution, as well as some testing facilities:

In [4]:
class AuthorisationConstraint:
    def __init__(self, instance, user, tasks):
        assert user >= 0 and user < instance.m
        assert all(t >= 0 and t < instance.n for t in tasks)

        self.user = user
        self.tasks = tasks

    def is_satisfied(self, solution):
        for task in set(range(solution.instance.n)) - set(self.tasks):
            if solution.assignment[task] == self.user:
                return False
        return True

    def write(self, f):
        f.write('Authorisations u' + str(self.user + 1))
        for t in self.tasks:
            f.write(' t' + str(t + 1))
        f.write('\n')


class BindingOfDutyConstraint:
    def __init__(self, instance, t1, t2):
        assert 0 <= t1 < instance.n
        assert t2 >= 0 and t2 < instance.n

        self.t1 = t1
        self.t2 = t2

    def is_satisfied(self, solution):
        return solution.assignment[self.t1] == solution.assignment[self.t2]

    def write(self, f):
        f.write('Binding-of-duty t%i t%i\n' % (self.t1 + 1, self.t2 + 1))


class SeparationOfDutyConstraint:
    def __init__(self, instance, t1, t2):
        assert 0 <= t1 < instance.n
        assert t2 >= 0 and t2 < instance.n

        self.t1 = t1
        self.t2 = t2

    def is_satisfied(self, solution):
        return solution.assignment[self.t1] != solution.assignment[self.t2]

    def write(self, f):
        f.write('Separation-of-duty t%i t%i\n' % (self.t1 + 1, self.t2 + 1))


class AdvancedConstraint1:
    def __init__(self, instance, k, tasks):
        assert 0 < k <= instance.n
        assert all(0 <= task <= instance.n for task in tasks)

        self.tasks = tasks
        self.k = k

    def is_satisfied(self, solution):
        return len(set(solution.assignment[task] for task in self.tasks)) <= self.k

    def write(self, f):
        f.write('AC1 %i' % self.k)
        for t in self.tasks:
            f.write(' t%i' % (t + 1))
        f.write('\n')


class AdvancedConstraint2:
    def __init__(self, instance, k, tasks):
        assert 1 <= k <= instance.n
        assert all(0 <= t < instance.n for t in tasks)

        self.tasks = tasks
        self.k = k

    def is_satisfied(self, solution):
        return len(set(solution.assignment[t] for t in self.tasks)) == self.k

    def write(self, f):
        f.write(f'AC2 {self.k} {" ".join(f"t{t+1}" for t in self.tasks)}\n')


class AdvancedConstraint3:
    def __init__(self, instance, tasks, teams):
        assert all(0 <= t < instance.n for t in tasks)
        assert all(0 <= team < len(instance.teams) for team in teams)

        self.tasks = tasks
        self.teams = teams
        self.instance = instance

    def is_satisfied(self, solution):
        return any(all(solution.assignment[t] in self.instance.teams[team] for t in self.tasks) for team in self.teams)

    def write(self, f):
        f.write(f'AC3 {" ".join(f"team{team+1}" for team in self.teams)} {" ".join(f"t{t+1}" for t in self.tasks)}\n')


class AdvancedConstraint4:
    def __init__(self, instance, team, k):
        assert 0 <= k < instance.n
        assert 0 <= team < len(instance.teams)

        self.team = team
        self.k = k
        self.instance = instance

    def is_satisfied(self, solution):
        return sum(1 for t in range(self.instance.n) if solution.assignment[t] in self.instance.teams[self.team]) <= self.k

    def write(self, f):
        f.write(f'AC4 team{self.team + 1} {self.k}\n')


class AdvancedConstraint5:
    def __init__(self, instance, team, supervisor):
        assert 0 <= team < len(instance.teams)
        assert 0 <= supervisor < instance.m

        self.team = team
        self.supervisor = supervisor
        self.instance = instance

    def is_satisfied(self, solution):
        return not any(solution.assignment[t] in self.instance.teams[self.team] for t in range(self.instance.n)) \
            or any(solution.assignment[t] == self.supervisor for t in range(self.instance.n))

    def write(self, f):
        f.write(f'AC5 team{self.team + 1} u{self.supervisor + 1}\n')


# Reads and stores instance data
class Instance:
    def __init__(self, filename):
        def parse_task(string):
            return int(re.match(r't(\d+)', string).group(1)) - 1

        def parse_user(string):
            return int(re.match(r'u(\d+)', string).group(1)) - 1

        def parse_team(string):
            return int(re.match(r'team(\d+)', string).group(1)) - 1

        if filename is None:
            return

        with open(filename, 'r') as f:
            import re
            self.n = int(re.match(r'^\s*#Tasks:\s+(\d+)\s*$', f.readline(), re.IGNORECASE).group(1))
            self.m = int(re.match(r'^\s*#Users:\s+(\d+)\s*$', f.readline(), re.IGNORECASE).group(1))

            t = int(re.match(r'^\s*#Teams:\s+(\d+)\s*$', f.readline(), re.IGNORECASE).group(1))
            self.teams = []
            for team_index in range(t):
                self.teams.append(list(map(parse_user, f.readline().strip().lower().split())))

            c = int(re.match(r'^\s*#Constraints:\s+(\d+)\s*$', f.readline(), re.IGNORECASE).group(1))

            self.constraints = []
            for line_index in range(c):
                line = f.readline().strip().lower()
                values = line.split()

                if values[0] == 'authorisations':
                    self.constraints.append(AuthorisationConstraint(
                        self, parse_user(values[1]), list(map(parse_task, values[2:]))))

                elif values[0] == 'binding-of-duty':
                    self.constraints.append(BindingOfDutyConstraint(self, parse_task(values[1]), parse_task(values[2])))

                elif values[0] == 'separation-of-duty':
                    self.constraints.append(SeparationOfDutyConstraint(self, parse_task(values[1]), parse_task(values[2])))

                elif values[0] == 'ac1':
                    self.constraints.append(AdvancedConstraint1(self, int(values[1]), list(map(parse_task, values[2:]))))

                elif values[0] == 'ac2':
                    self.constraints.append(AdvancedConstraint2(self, int(values[1]), list(map(parse_task, values[2:]))))

                elif values[0] == 'ac3':
                    teams = []
                    index = 1
                    while values[index].startswith('team'):
                        teams.append(parse_team(values[index]))
                        index += 1

                    self.constraints.append(AdvancedConstraint3(self, list(map(parse_task, values[index:])), teams))

                elif values[0] == 'ac4':
                    self.constraints.append(AdvancedConstraint4(self, parse_team(values[1]), int(values[2])))

                elif values[0] == 'ac5':
                    self.constraints.append(AdvancedConstraint5(self, parse_team(values[1]), parse_user(values[2])))

                else:
                    raise Exception(f'Unknown constraint {values[0]}.')

    def save(self, filename):
        import os, sys
        with open(filename, 'w') as f:
            f.write('#Tasks: ' + str(self.n) + '\n')
            f.write('#Users: ' + str(self.m) + '\n')
            f.write('#Teams: ' + str(len(self.teams)) + '\n')

            for team in self.teams:
                f.write(' '.join(f'u{u+1}' for u in team))
                f.write('\n')

            f.write('#Constraints: ' + str(len(self.constraints)) + '\n')
            for c in self.constraints:
                c.write(f)


# Stores a solution to a WSP instance
class Solution:
    def __init__(self, instance, sat):
        self.instance = instance
        self.sat = sat
        self.assignment = [-1]*self.instance.n

    # Use this function to specify that user 'user' is assigned to task 'task'
    def assign_user(self, task, user):
        if task < 0 or task >= self.instance.n:
            raise Exception(f'Task {task} is outside the range.')

        if user < 0 or user >= self.instance.m:
            raise Exception(f'User {user} is outside the range.')

        self.assignment[task] = user




def ensure_instances_downloaded():
    from os.path import exists

    if not exists('instances.zip'):
        print(f'Downloading \'instances.zip\'...')

        url = 'https://www.dropbox.com/scl/fi/nsf3vmnwipxw6ueoc2hch/test-instances.zip?rlkey=ateai2kbwf0mg2njof0vxer4f&dl=1'

        import urllib
        req = urllib.request.Request(url)

        with urllib.request.urlopen(req) as file:
            with open('instances.zip', 'wb') as f:
                f.write(file.read())

        print(f'Unpacking...')
        import zipfile
        with zipfile.ZipFile('instances.zip', 'r') as zip_ref:
            zip_ref.extractall('.')

        print(f'Instances are ready')


def run_test(filename, known_to_be_sat):
    def coloured_print(text, colour):
        from IPython.core.display import display, HTML
        display(HTML(f'<span style=color:{colour}><pre>{text}</pre></span>'))

    ensure_instances_downloaded()

    instance = Instance(filename)

    import time

    starttime = time.perf_counter()
    solution = solve(instance)
    endtime = time.perf_counter()

    def print_test_header(passed):
        coloured_print(f'{filename:<30} {"  (sat)" if known_to_be_sat else "(unsat)"} {(endtime - starttime) * 1000:5.0f} ms {"PASS" if passed else "FAIL"}', 'green' if passed else 'red')

    if solution.sat:
        if len(solution.assignment) != instance.n or not all(0 <= user <= instance.m for user in solution.assignment):
            print_test_header(False)
            print('  assignment of users to tasks is infeasible.')
            return float('inf')

        broken = [c for c in instance.constraints if not c.is_satisfied(solution)]
        if len(broken) > 0:
            print_test_header(False)
            print('  the solution breaks some constraints:')
            for t in [AuthorisationConstraint, BindingOfDutyConstraint, SeparationOfDutyConstraint, AdvancedConstraint1, AdvancedConstraint2, AdvancedConstraint3, AdvancedConstraint4, AdvancedConstraint5]:
                print(f'    {len([0 for c in broken if isinstance(c, t)])} broken {t.__name__} constraints')
            return float('inf')

    correct = solution.sat == known_to_be_sat
    if correct:
        print_test_header(True)
    else:
        print_test_header(False)
        print(f'  Expected outcome:  {"sat" if known_to_be_sat else "unsat"}')
        print(f'    Actual outcome:  {"sat" if solution.sat else "unsat"}')

    return endtime - starttime


ensure_instances_downloaded()


def test_batch(batch_index: int):
    print(f'Testing batch{batch_index}:')
    total = 0.0
    for i in range(1, 11):
        total += run_test(f'batch{batch_index}/inst_{i}.txt', i % 2 == 1)

    print(f'Total time for batch{batch_index}: {total:0.1f} sec\n')

Unpacking...
Instances are ready


You are expected to implement function `solve(instance)` that takes an object of class `Instance` as a parameter and returns an object of class `Solution`.

You can extend the functionality of the provided classes as you wish.  If you change those classes, please include them into your submission.

Implement the `solve(instance)` function below.

In [ ]:
from ortools.sat.python import cp_model

def solve(instance):
    model = cp_model.CpModel()
    n = instance.n  # number of tasks
    m = instance.m  # number of users

    # Decision variables: x[t,u] = 1 if task t is assigned to user u
    x = {(t, u): model.NewBoolVar(f'x_{t}_{u}') for t in range(n) for u in range(m)}

    # Each task must be assigned to exactly one user
    for t in range(n):
        model.AddExactlyOne(x[t, u] for u in range(m))

    # Process constraints
    for c in instance.constraints:
        # Authorisation: user can only do authorized tasks
        if isinstance(c, AuthorisationConstraint):
            for t in range(n):
                if t not in c.tasks:
                    model.Add(x[t, c.user] == 0)

        # Binding of duty: two tasks must be assigned to the same user
        elif isinstance(c, BindingOfDutyConstraint):
            # Optimized: instead of m constraints, use one constraint per user pair
            for u in range(m):
                model.Add(x[c.t1, u] == x[c.t2, u])

        # Separation of duty: two tasks must be assigned to different users
        elif isinstance(c, SeparationOfDutyConstraint):
            # Optimized: instead of m constraints, use direct constraint
            for u in range(m):
                model.Add(x[c.t1, u] + x[c.t2, u] <= 1)

        # AC1: At most k different users for the given tasks
        elif isinstance(c, AdvancedConstraint1):
            user_used = []
            for u in range(m):
                b = model.NewBoolVar(f"ac1_user_{id(c)}_{u}")
                # b is true iff user u is assigned at least one task from c.tasks
                model.AddMaxEquality(b, [x[t, u] for t in c.tasks])
                user_used.append(b)
            model.Add(sum(user_used) <= c.k)

        # AC2: Exactly k different users for the given tasks
        elif isinstance(c, AdvancedConstraint2):
            user_used = []
            for u in range(m):
                b = model.NewBoolVar(f"ac2_user_{id(c)}_{u}")
                # b is true iff user u is assigned at least one task from c.tasks
                model.AddMaxEquality(b, [x[t, u] for t in c.tasks])
                user_used.append(b)
            model.Add(sum(user_used) == c.k)

        # AC3: All tasks assigned to members of exactly one team from the list
        elif isinstance(c, AdvancedConstraint3):
            # For each team, create a boolean indicating if this team is selected
            team_selected = []
            for team_idx in c.teams:
                team_bool = model.NewBoolVar(f"ac3_team_{id(c)}_{team_idx}")
                team_selected.append(team_bool)

                # If this team is selected, all tasks must be done by this team
                team_users = instance.teams[team_idx]
                for t in c.tasks:
                    model.Add(sum(x[t, u] for u in team_users) == 1).OnlyEnforceIf(team_bool)

            # Exactly one team must be selected
            model.AddExactlyOne(team_selected)

        # AC4: Team workload limit - at most k tasks assigned to team members
        elif isinstance(c, AdvancedConstraint4):
            team_users = instance.teams[c.team]
            model.Add(sum(x[t, u] for t in range(n) for u in team_users) <= c.k)

        # AC5: If any team member is assigned a task, supervisor must be assigned too
        elif isinstance(c, AdvancedConstraint5):
            team_users = instance.teams[c.team]
            supervisor = c.supervisor

            # Create boolean for whether team has any task
            team_has_task = model.NewBoolVar(f"ac5_team_{id(c)}")
            model.AddMaxEquality(team_has_task, [x[t, u] for t in range(n) for u in team_users])

            # Create boolean for whether supervisor has any task
            supervisor_has_task = model.NewBoolVar(f"ac5_super_{id(c)}")
            model.AddMaxEquality(supervisor_has_task, [x[t, supervisor] for t in range(n)])

            # If team has task, supervisor must have task
            model.AddImplication(team_has_task, supervisor_has_task)

    # Solver configuration for better performance
    solver = cp_model.CpSolver()
    solver.parameters.num_search_workers = 8  # Parallel search
    solver.parameters.max_time_in_seconds = 60.0  # 60 second timeout per instance
    solver.parameters.log_search_progress = False  # Reduce output

    status = solver.Solve(model)

    sat = status in (cp_model.OPTIMAL, cp_model.FEASIBLE)
    sol = Solution(instance, sat)

    if sat:
        for t in range(n):
            for u in range(m):
                if solver.Value(x[t, u]) == 1:
                    sol.assign_user(t, u)
                    break

    return sol

Run this cell to test your `solve(instance)` function.

In [3]:
for batch_index in range(1, 10):
    test_batch(batch_index)

NameError: name 'test_batch' is not defined